課題1：【データクレンジング＆エラーハンドリング】（SQL / Polars）
〜本業の名寄せ・ラベリング業務のスピードを爆上げする修行〜

ビジネス背景：
データサイエンスチームから「クラスタリングのモデルにデータを投入したいが、欠損値や異常値が含まれているため、BIやAIが誤作動を起こさないよう、データエンジニアリング側で綺麗なデータに整形してほしい」と依頼された。

タスク：

ビジネスロジックに基づく欠損値補正：
MINIMUM_PAYMENTS（最低支払い金額）の欠損値を、単なる「0」や全体の平均値で埋めるのではなく、「その会員の BALANCE（残高）の10%」または「同じ TENURE（保有期間）の会員の平均値」で動的に補正するロジックを実装せよ。

異常値の検知とラベリング：
クレジットカード限度額（CREDIT_LIMIT）に対して、残高（BALANCE）が超えている（＝オーバートップしている）異常な会員を検知し、is_over_limit フラグ（1または0）を立てよ。

文字列の正規化（名寄せへの応用）：
CUST_ID の前後に不要なスペースや改行コードが含まれていると仮定し（実務の汚いデータを想定）、これらを綺麗にトリム（除去）した一意のIDを生成せよ。

Intermediate（中級）として意識すべき裏側の動き：

これをPolarsの LazyFrame（遅延評価）を使い、メモリを極限まで節約したベクトル演算（when().then().otherwise()）で、一撃かつ高速に処理するコードが書けるか。


In [ ]:
# データの取得
from snowflake.snowpark.context import get_active_session
import polars as pl 

session = get_active_session()

df_pd = (
    session
        .sql("SELECT * FROM KAGGLE_CREDIT_CARD.PUBLIC.RAW")
        .to_pandas()
)
df_pl = pl.DataFrame(df_pd)
print(df_pl.shape)
display(df_pl.head())


In [ ]:
# 全体の欠損があるのかを確認する。
print('DFの欠損があるのかを確認する。')
display(df_pl.null_count())

print('-' * 100)

# MINIMUM_PAYMENTSとCREDIT_LIMITの欠損をそれぞれ確認する
print('MINIMUM_PAYMENTSが欠損')
display(df_pl.filter(pl.col("MINIMUM_PAYMENTS").is_null()))
print('CREDIT_LIMITが欠損')
display(df_pl.filter(pl.col("CREDIT_LIMIT").is_null()))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

df_pl = (
    df_pl.with_columns(
        pl.when(pl.col("MINIMUM_PAYMENTS").is_null()).then(1).otherwise(0).alias('IS_NULL_MINIMUM_PAYMENTS')
    )
)
df_pd = df_pl.to_pandas()

# sebornのdensity : 横幅と縦幅の決め方
# 棒が50個になるようにする、
# 仮に各区間が1000, 全員を100人、10名が最初の棒にいるとすると、
# 横幅は1000
# 縦幅は、10(人) / 100(人) / 1000(幅) 
# => 対象人数の割合をさらに区間幅で割ることで合計の数が1になるようにする
g = sns.histplot(
    data = df_pd,
    x = 'BALANCE', 
    hue='IS_NULL_MINIMUM_PAYMENTS',
    bins=50,
    stat= 'density', 
    common_norm = False,
    kde=True,
    alpha = 0.4,
    element="step"
)
plt.show()

* 共通点 : 初期値にボリュームゾーンがある。
* NULLは比較的0側にボリュームゾーンがあり右側の裾が短い
* 値ありユーザは高額決済のところにもボリュームがあり右側の裾が長い

結論、全体の平均値で埋めると値ありユーザの値に引っ張られてしまう可能性がある、
一方、慣例に従い、NULL側のユーザの下側10%値を用いれば、ボリュームゾーンの値で自然なnull値の埋め込みになることを期待できる。

In [ ]:
# 下側10%の値で埋める場合
df_tmp = df_pl.clone()
q10 = df_tmp.filter(pl.col("IS_NULL_MINIMUM_PAYMENTS")==1).select("BALANCE").quantile(0.1).to_series()[0]
print(q10)

df_tmp = (
    df_pl.with_columns(
        pl.col("MINIMUM_PAYMENTS").fill_null(q10).alias("FULLFILLED_TGT")
    )
)

# TENUREごとのBALANCE平均値で埋める場合
df_tenure_mean = (
        df_pl
        .filter(pl.col("MINIMUM_PAYMENTS").is_null())
        .group_by("TENURE")
        .agg(
            pl.col("BALANCE").mean().alias("MEAN_BALANCE"))
)

display(df_tenure_mean.sort(by="TENURE"))

# tmp_2に対して、上記の集計値で埋め込みをする
df_tmp_2 = df_pl.clone()
df_tmp_2 = (
    df_tmp_2.join(df_tenure_mean, on=["TENURE"], how="left")
)
df_tmp_2 = (
    df_tmp_2.with_columns(
        pl
        .when(pl.col("MINIMUM_PAYMENTS").is_null() )
        .then(pl.col("MEAN_BALANCE"))
        .otherwise(pl.col("BALANCE"))
        .alias("FULLFILLED_TGT")
    )
)

display(
    df_tmp_2
    .filter(pl.col("MINIMUM_PAYMENTS").is_null())
    .select(pl.col(["TENURE", "MINIMUM_PAYMENTS", "FULLFILLED_TGT","MEAN_BALANCE"]))
    .unique().sort(by="TENURE")
)

補正方法を2つ準備したがどちらが正しいのかを調べるため、\
欠損値がもたらすデメリットを把握し、欠損値を埋めることのメリットと目的を再確認する。

### 欠損値のデメリット
* AI, BIでのエラー, 偏りのある予測結果が出てくる
* 欠損値を削除すると解析に必要な情報量が減ってしまう
### 欠損値を埋めることのメリット, 目的
* 上記のデメリットを回避し、情報量を適正な量保持したままAI, BIに渡すことができる
* 目的 : 情報量を保持しつつも、従来のデータの偏りを変化させない方法でnull値を埋めたい

In [ ]:
# null値を埋めた2手法の中で、データの元々の偏りへの影響が少ないものを選定する, データの偏りは相関係数で表すことができる
# 今回のnull値を埋めた対象はMINIMUM_PAYMENTS(クレジット会社への最低限の支払い金額, これ以上払わないとカード使えない)であり、
# これはBALANCE(クレジット会社への支払い総額)の金額と強い相関があることが既知であることから、
# MINIMUM_PAYMENTSとBALANCEの相関係数が、null値埋め込み前後でどれくらい変わるのかを調べる

print("元々のデータの相関係数")
display(df_pl.filter(~pl.col("MINIMUM_PAYMENTS").is_null()).select(pl.corr("BALANCE", "MINIMUM_PAYMENTS")).to_series()[0])
print("手法1 : 下側10%の値でのnull補完")
display(df_tmp.select(pl.corr("BALANCE", "FULLFILLED_TGT")).to_series()[0])
print("手法2 : TENUREごと平均値のnull補完")
display(df_tmp_2.select(pl.corr("BALANCE", "FULLFILLED_TGT")).to_series()[0])

In [ ]:
# 上記の結果、手法1の方が元々の相関関係に影響が少ないので手法1を採択する。
# 一方、念の為他の値に対するMINIMUM_PAYMENTSとの相関関係を一覧で算出し処理前後での影響度合いを確認する

# FULLFILLED_TGTをMINIMUM_PAYMENTSに置換
df_tmp = df_tmp.with_columns(pl.col("FULLFILLED_TGT").alias("MINIMUM_PAYMENTS")).select(pl.exclude("FULLFILLED_TGT"))

In [ ]:
# MINIMUM_PAYMENTS以外の数値型(int, float)のカラム名を取得する
import polars.selectors as cs
columns = df_tmp.select(cs.numeric()).select(pl.exclude("MINIMUM_PAYMENTS")).columns

def corr_check(columns_list, df_raw, df_tgt):
    df_raw = df_raw.filter(~pl.col("MINIMUM_PAYMENTS").is_null())
    
    
    for i in columns_list:
        print(i)
        print("BEFORE")
        display(df_raw.select(pl.corr("MINIMUM_PAYMENTS", i)).to_series()[0])
        print("AFTER")
        display(df_tmp.select(pl.corr("MINIMUM_PAYMENTS", i)).to_series()[0])
        print('-'*100)
    

In [ ]:
corr_check(columns, df_pl, df_tmp)

In [ ]:
# クレジットカード限度額（CREDIT_LIMIT）に対して、残高（BALANCE）が超えている（＝オーバートップしている）異常な会員を検知し、
# is_over_limit フラグ（1または0）を立てよ。
df_pl = df_tmp.clone()
df_pl.head()

In [ ]:
df_pl = (
    df_pl.with_columns(
        pl.when(pl.col("BALANCE") >= pl.col("CREDIT_LIMIT")).then(1).otherwise(0).alias("IS_OVER_LIMIT")
    )
)
df_pl.head()

In [ ]:
df_pl.columns

In [ ]:
%%sql -r dataframe_3
-- CREATE THE PROCESSED NEW TABLE
CREATE OR ALTER TABLE KAGGLE_CREDIT_CARD.PUBLIC.OUTPUT_1(
    CUST_ID STRING,
    BALANCE FLOAT,
    BALANCE_FREQUENCY FLOAT,
    PURCHASES FLOAT,
    ONEOFF_PURCHASES FLOAT,
    INSTALLMENTS_PURCHASES FLOAT,
    CASH_ADVANCE FLOAT,
    PURCHASES_FREQUENCY FLOAT,
    ONEOFFPURCHASESFREQUENCY FLOAT,
    PURCHASESINSTALLMENTSFREQUENCY FLOAT,
    CASHADVANCEFREQUENCY FLOAT,
    CASHADVANCETRX FLOAT,
    PURCHASES_TRX FLOAT,
    CREDIT_LIMIT FLOAT,
    PAYMENTS FLOAT,
    MINIMUM_PAYMENTS FLOAT,
    PRCFULLPAYMENT FLOAT,
    TENURE INT,
    IS_NULL_MINIMUM_PAYMENTS INT,
    IS_OVER_LIMIT INT
)

In [ ]:
df_pd = df_pl.to_pandas()

session.write_pandas(
    df = df_pd,
    table_name = 'OUTPUT_1',
    database = 'KAGGLE_CREDIT_CARD',
    schema = 'PUBLIC',
    overwrite = True,
    auto_create_table = False
)